# 🧹 Amazon Orders Data Pipeline (2020–2024)
## 🧩 Notebook 02 — Data Cleaning

---

## 🔗 Dependency

This notebook depends on:

%run /Workspace/Amazon_project/01_data_ingestion_and_validation

Notebook 01 performs:
- Environment initialization
- Schema enforcement
- Structural validation
- Business rule validation

This notebook focuses on transforming validated raw data into a clean analytical dataset.

---

## 🎯 Objective

The goal of this notebook is to:

1. Handle missing values
2. Remove duplicates
3. Correct invalid records
4. Standardize categorical values
5. Normalize text fields
6. Prepare a clean dataset for EDA and feature engineering

The output of this notebook is a cleaned DataFrame.

In [0]:
%run /Workspace/Amazon_project/01_data_ingestion_and_validation

Successfully loaded 100000 records from Bronze Layer.


OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0,0.0,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.1,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


### 🔹 Safety Check

In [0]:
assert 'df' in globals(), "Run Notebook 01 first."

### 🔹 Remove Duplicates

In [0]:
df = df.dropDuplicates(["OrderID"])

### 🔹 Handle Missing Values

In [0]:
critical_columns = ["OrderID", "OrderDate", "CustomerID", "ProductID"]

rows_before = df.count()
df = df.dropna(subset=critical_columns)
rows_after = df.count()

print("Rows removed:", rows_before - rows_after)

Rows removed: 0


In [0]:
df = df.fillna({
    "Discount": 0,
    "ShippingCost": 0,
    "Tax": 0
})

### 🔹  Fix Invalid Numeric Values

In [0]:
df = df.filter("Quantity > 0")
df = df.filter("UnitPrice >= 0")
df = df.filter("Tax >= 0")
df = df.filter("ShippingCost >= 0")

In [0]:
df_validated = df.withColumn(
    "CalculatedTotal", 
    f.round(f.col("UnitPrice") * f.col("Quantity")*(1- f.col("Discount")) + f.col("ShippingCost") + f.col("Tax") , 2)
)

df_validated = df_validated.withColumn(
    "AmountVariance", f.round(f.abs(f.col("TotalAmount") - f.col("CalculatedTotal")), 2)
).withColumn(
    "IsAmountValid", 
    f.when(f.col("AmountVariance") <= 0.02, True).otherwise(False)
)

invalid_count = df_validated.filter(f.col("IsAmountValid") == False).count()
total_count = df_validated.count()

print(f"Total Records: {total_count}")
print(f"Integrity Failures: {invalid_count} ({(invalid_count/total_count)*100:.2f}%)")


if invalid_count > 0:
    print("Reviewing discrepant records (Top 10):")
    display(
        df_validated.filter(f.col("IsAmountValid") == False)
        .select("OrderID", "UnitPrice", "Quantity", "TotalAmount", "CalculatedTotal", "AmountVariance")
        .limit(10)
    )

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7282507561705463>, line 3
      1 df_validated = df.withColumn(
      2     "CalculatedTotal", 
----> 3     f.round(f.col("UnitPrice") * f.col("Quantity")*(1- f.col("Discount")) + f.col("ShippingCost") + f.col("Tax") , 2)
      4 )
      6 df_validated = df_validated.withColumn(
      7     "AmountVariance", f.round(f.abs(f.col("TotalAmount") - f.col("CalculatedTotal")), 2)
      8 ).withColumn(
      9     "IsAmountValid", 
     10     f.when(f.col("AmountVariance") <= 0.02, True).otherwise(False)
     11 )
     13 invalid_count = df_validated.filter(f.col("IsAmountValid") == False).count()

NameError: name 'f' is not defined

In [0]:

df = df.withColumn("OrderDate", f.to_date(f.col("OrderDate")))

### 🔹 Standardize Categorical Columns

In [0]:
text_columns = ["Category","OrderStatus","City","State"]

for col_name in text_columns:
    df = df.withColumn(
        col_name,
        f.initcap(f.trim(f.col(col_name)))
    )

name_columns = ["Brand","ProductName","CustomerName"]

for col_name in name_columns:
    df = df.withColumn(
        col_name,
        f.trim(f.col(col_name))
    )

df = df.withColumn("PaymentMethod", f.upper(f.trim(f.col("PaymentMethod"))))
df = df.withColumn("Country", f.upper(f.trim(f.col("Country"))))

In [0]:
display(df.limit(10))

## 📊 Cleaning Summary

The following cleaning steps were performed:

- Duplicate orders removed
- Critical null records dropped
- Non-critical null values imputed
- Invalid numeric values filtered
- Categorical values standardized
- Date range validated

The resulting clean dataset is now ready for exploratory data analysis.